In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error

INPUT_SEQUENCE_LENGTH = 24
OUTPUT_SEQUENCE_LENGTH = 24 # Predict the next 24 hours
NUM_FEATURES = 9

# Reduced batch size to accommodate higher VRAM overhead of Encoder-Decoder
batch_size = 2048 
print(f"batch size: {batch_size}")

# =====================================================
# LOAD COLUMNS
# =====================================================
required_cols = [
    'timestamp', 'state_code', 'county_code', 'site_num',
    'latitude', 'longitude', 'temperature', 'wind_speed', 'pm25_level'
]

backup_df = pd.read_parquet(
    "/kaggle/input/datasets/flixrojas/usa-2018-2021-aq-df-geoquery-parquet",
    columns=required_cols
)

# downcast to float32
float_cols = ['latitude', 'longitude', 'temperature', 'wind_speed', 'pm25_level']
backup_df[float_cols] = backup_df[float_cols].astype('float32')

# Convert grouping columns to category dtype to save memory
group_cols = ['state_code', 'county_code', 'site_num']
for col in group_cols:
    backup_df[col] = backup_df[col].astype('category')

# =====================================================
# CALCULATE THRESHOLDS
# =====================================================
historical_mean = backup_df['pm25_level'].mean()
std_dev = backup_df['pm25_level'].std()
mad_underfit_threshold = (backup_df['pm25_level'] - historical_mean).abs().mean()

# Sort in-place for persistence calculation
backup_df.sort_values(by=group_cols + ['timestamp'], inplace=True)
backup_df['pm25_shifted'] = backup_df.groupby(group_cols, observed=True)['pm25_level'].shift(1)
persistence_mae = (backup_df['pm25_level'] - backup_df['pm25_shifted']).abs().mean()
backup_df.drop(columns=['pm25_shifted'], inplace=True)  # clean up

overfit_gap_threshold = std_dev * 0.15

print(f"--- Dataset PM2.5 Statistics ---")
print(f"Historical Mean:      {historical_mean:.2f} µg/m³")
print(f"Standard Deviation:   {std_dev:.2f} µg/m³")
print(f"Underfitting (MAD):   {mad_underfit_threshold:.2f} µg/m³")
print(f"Underfitting (Persistence): {persistence_mae:.2f} µg/m³")
print(f"Overfitting Gap:      {overfit_gap_threshold:.2f} µg/m³")

# =====================================================
# FEATURE ENGINEERING (float32)
# =====================================================
day_of_year = backup_df['timestamp'].dt.dayofyear.astype('float32')
hour_of_day = backup_df['timestamp'].dt.hour.astype('float32')

DAYS_IN_YEAR = 365.2425
HOURS_IN_DAY = 24.0

backup_df['day_sin']  = np.sin(2 * np.pi * day_of_year / DAYS_IN_YEAR).astype('float32')
backup_df['day_cos']  = np.cos(2 * np.pi * day_of_year / DAYS_IN_YEAR).astype('float32')
backup_df['hour_sin'] = np.sin(2 * np.pi * hour_of_day / HOURS_IN_DAY).astype('float32')
backup_df['hour_cos'] = np.cos(2 * np.pi * hour_of_day / HOURS_IN_DAY).astype('float32')

# =====================================================
# CHRONOLOGICAL SPLIT
# =====================================================
unique_times = backup_df['timestamp'].sort_values().unique()
split_time = unique_times[int(len(unique_times) * 0.8)]

train_df = backup_df[backup_df['timestamp'] < split_time].copy()
val_df   = backup_df[backup_df['timestamp'] >= split_time].copy()

# =====================================================
# FIT SCALERS (float64 for fitting, float32 for data)
# =====================================================
lstm_features = [
    'pm25_level', 'latitude', 'longitude', 'temperature', 'wind_speed',
    'day_sin', 'day_cos', 'hour_sin', 'hour_cos'
]
target_col = 'pm25_level'

scaler_x = StandardScaler()
scaler_y = MinMaxScaler()

# Fit on train (scaler requires float64 for precision)
scaler_y.fit(train_df[[target_col]].astype('float64'))
scaler_x.fit(train_df[lstm_features].astype('float64'))

# Transform and cast back to float32 to save memory
train_df[lstm_features] = scaler_x.transform(train_df[lstm_features]).astype('float32')
val_df[lstm_features]   = scaler_x.transform(val_df[lstm_features]).astype('float32')

joblib.dump(scaler_x, 'scaler_x_seq2seq.pkl')
joblib.dump(scaler_y, 'scaler_y_seq2seq.pkl')

# ====================================
# SEQUENCE CREATION DIRECTLY TO DISK 
# ====================================
def count_sequences(df, in_seq_len, out_seq_len):
    """Count total valid sequences without building them."""
    total = 0
    grouped = df.groupby(['latitude', 'longitude'], observed=True)
    for _, group in grouped:
        total += max(0, len(group) - in_seq_len - out_seq_len + 1)
    return total

def create_seq2seq_mmap(df, features, target, in_seq_len, out_seq_len, out_X_path, out_y_path):
    """
    Build Seq2Seq LSTM sequences and write directly to memory-mapped files on disk.
    """
    n_samples = count_sequences(df, in_seq_len, out_seq_len)
    if n_samples == 0:
        raise ValueError("No sequences can be created from the provided DataFrame.")

    n_features = len(features)

    # Create memory-mapped files (write mode)
    X_mmap = np.memmap(out_X_path, dtype='float32', mode='w+',
                       shape=(n_samples, in_seq_len, n_features))
    y_mmap = np.memmap(out_y_path, dtype='float32', mode='w+',
                       shape=(n_samples, out_seq_len, 1))

    idx = 0
    grouped = df.groupby(['latitude', 'longitude'], observed=True)
    for _, group in grouped:
        group = group.sort_values('timestamp')
        feat = group[features].values
        targ = group[target].values

        n_seqs = len(group) - in_seq_len - out_seq_len + 1
        if n_seqs <= 0:
            continue

        for i in range(n_seqs):
            X_mmap[idx] = feat[i : i + in_seq_len]
            # Slicing the target to fetch 'out_seq_len' timesteps
            y_mmap[idx] = targ[i + in_seq_len : i + in_seq_len + out_seq_len].reshape(-1, 1)
            idx += 1

    del X_mmap
    del y_mmap

    return n_samples

# Build sequences and save to disk
print("Creating training sequences on disk...")
train_samples = create_seq2seq_mmap(
    train_df, lstm_features, target_col, INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH,
    'X_train_seq2seq.dat', 'y_train_seq2seq.dat'
)

print("Creating validation sequences on disk...")
val_samples = create_seq2seq_mmap(
    val_df, lstm_features, target_col, INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH,
    'X_val_seq2seq.dat', 'y_val_seq2seq.dat'
)

# =====================================================
print("# FREE DATAFRAME RAM (no longer needed)")
# =====================================================
del train_df, val_df, backup_df

# =====================================================
# SEQ2SEQ MODEL ARCHITECTURE
# =====================================================
def build_seq2seq_model(in_seq_len, out_seq_len, num_features):
    # Encoder
    encoder_inputs = layers.Input(shape=(in_seq_len, num_features))
    encoder_lstm = layers.LSTM(64, return_state=True)
    encoder_outputs, state_h, state_c = encoder_lstm(encoder_inputs)

    # The encoder final state provides the context for the decoder
    encoder_states = [state_h, state_c]

    # Decoder
    # Repeat the context vector 'out_seq_len' times for the decoder inputs
    decoder_inputs = layers.RepeatVector(out_seq_len)(encoder_outputs)
    
    # Return full sequences so we get a prediction per timestep
    decoder_lstm = layers.LSTM(64, return_sequences=True)
    decoder_outputs = decoder_lstm(decoder_inputs, initial_state=encoder_states)

    # TimeDistributed Dense layer applies the dense operation to every timestep individually
    decoder_dense = layers.TimeDistributed(layers.Dense(1))
    outputs = decoder_dense(decoder_outputs)

    model = models.Model(inputs=encoder_inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    return model

seq2seq_model = build_seq2seq_model(INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH, NUM_FEATURES)
seq2seq_model.summary()

# =====================================================
# GENERATORS AND MMAP LOADING
# =====================================================
def get_mmap_shape(file_path, samples, in_seq=None, out_seq=None, n_feats=None, is_X=True):
    if is_X:
        return (samples, in_seq, n_feats)
    else:
        return (samples, out_seq, 1)

train_X_shape = get_mmap_shape('X_train_seq2seq.dat', train_samples, in_seq=INPUT_SEQUENCE_LENGTH, n_feats=NUM_FEATURES, is_X=True)
train_y_shape = get_mmap_shape('y_train_seq2seq.dat', train_samples, out_seq=OUTPUT_SEQUENCE_LENGTH, is_X=False)
val_X_shape   = get_mmap_shape('X_val_seq2seq.dat', val_samples, in_seq=INPUT_SEQUENCE_LENGTH, n_feats=NUM_FEATURES, is_X=True)
val_y_shape   = get_mmap_shape('y_val_seq2seq.dat', val_samples, out_seq=OUTPUT_SEQUENCE_LENGTH, is_X=False)

X_train_mmap = np.memmap('X_train_seq2seq.dat', dtype='float32', mode='r', shape=train_X_shape)
y_train_mmap = np.memmap('y_train_seq2seq.dat', dtype='float32', mode='r', shape=train_y_shape)
X_val_mmap   = np.memmap('X_val_seq2seq.dat', dtype='float32', mode='r', shape=val_X_shape)
y_val_mmap   = np.memmap('y_val_seq2seq.dat', dtype='float32', mode='r', shape=val_y_shape)

def data_generator(X_mmap, y_mmap, batch_size):
    """Yields batches from memory-mapped arrays without loading everything."""
    n_samples = X_mmap.shape[0]
    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        yield X_mmap[start:end], y_mmap[start:end]

output_signature = (
    tf.TensorSpec(shape=(None, INPUT_SEQUENCE_LENGTH, NUM_FEATURES), dtype=tf.float32),
    tf.TensorSpec(shape=(None, OUTPUT_SEQUENCE_LENGTH, 1), dtype=tf.float32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(X_train_mmap, y_train_mmap, batch_size),
    output_signature=output_signature
).prefetch(tf.data.AUTOTUNE).repeat() 

val_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(X_val_mmap, y_val_mmap, batch_size),
    output_signature=output_signature
).prefetch(tf.data.AUTOTUNE)

# =====================================================
# MODEL TRAINING
# =====================================================
steps_per_epoch = train_samples // batch_size

checkpoint = ModelCheckpoint('seq2seq_pm25_model.keras', save_best_only=True, monitor='val_loss')
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("Training Seq2Seq Model...")
seq2seq_model.fit(
    train_dataset,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=val_dataset,
    callbacks=[checkpoint, early_stop],
    verbose=1
)

# =====================================================
# EVALUATION
# =====================================================
print("Generating predictions on validation set via generator...")
y_pred_scaled = seq2seq_model.predict(val_dataset, verbose=1)
y_true_scaled = y_val_mmap[:] 

# Inverse transform (Reshaping back to 2D for the scaler, then back to 3D/Flattening)
scaler_y = joblib.load('scaler_y_seq2seq.pkl')

y_pred_flat = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_true_flat = scaler_y.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()

# Compute overall metrics across all sequence steps
mae = mean_absolute_error(y_true_flat, y_pred_flat)
rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))

print(f"MAE:  {mae:.2f} µg/m³")
print(f"RMSE: {rmse:.2f} µg/m³")
print(f"Baseline Persistence MAE: {persistence_mae:.2f} µg/m³")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.cloud import bigquery
from sklearn.metrics import mean_absolute_error, mean_squared_error

def find_reliable_sensor_sites(client, year):
    print(f"Scanning EPA database for sites with all 3 sensors in {year}...")
    
    diagnostic_query = f"""
    SELECT
        pm.state_code,
        pm.county_code,
        pm.site_num,
        COUNT(*) AS concurrent_hours
    FROM
        `bigquery-public-data.epa_historical_air_quality.pm25_frm_hourly_summary` AS pm
    INNER JOIN
        `bigquery-public-data.epa_historical_air_quality.temperature_hourly_summary` AS t
        ON pm.state_code = t.state_code
        AND pm.county_code = t.county_code
        AND pm.site_num = t.site_num
        AND pm.date_local = t.date_local
        AND pm.time_local = t.time_local
    INNER JOIN
        `bigquery-public-data.epa_historical_air_quality.wind_hourly_summary` AS w
        ON pm.state_code = w.state_code
        AND pm.county_code = w.county_code
        AND pm.site_num = w.site_num
        AND pm.date_local = w.date_local
        AND pm.time_local = w.time_local
    WHERE
        pm.date_local >= '{year}-01-01'
        AND pm.date_local <= '{year}-12-31'
    GROUP BY
        pm.state_code,
        pm.county_code,
        pm.site_num
    ORDER BY
        concurrent_hours DESC
    LIMIT 10;
    """
    
    query_job = client.query(diagnostic_query)
    best_sites_df = query_job.to_dataframe()
    
    return best_sites_df

def evaluate_epa_recursive_multi_day(client, model, scaler_x, scaler_y, state_code, county_code, site_num, start_date, days=14):
    """
    Performs a closed-loop recursive forecast over a multi-day window using EPA data.
    The model predicts step t+1, and that prediction is fed back into the sequence 
    to predict step t+2, flying blind on PM2.5 but using true future weather.
    """
    # 1. Define the exact time window
    start_time = pd.to_datetime(start_date)
    end_time = start_time + pd.Timedelta(days=days)
    
    # We require exactly 7 true historical hours to seed the first prediction
    buffer_time = start_time - pd.Timedelta(hours=7) 
    
    print(f"Querying EPA BigQuery from {buffer_time} to {end_time}...")
    
    # 2. Configure parameterized query
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("state_code", "STRING", state_code),
            bigquery.ScalarQueryParameter("county_code", "STRING", county_code),
            bigquery.ScalarQueryParameter("site_num", "STRING", site_num),
            bigquery.ScalarQueryParameter("buffer_time", "TIMESTAMP", buffer_time),
            bigquery.ScalarQueryParameter("end_time", "TIMESTAMP", end_time),
        ]
    )
    
    # 3. SQL Query with strict timestamp parsing
    query = """
    WITH raw_epa_data AS (
        SELECT
            PARSE_TIMESTAMP('%Y-%m-%d %H:%M', CONCAT(CAST(pm.date_local AS STRING), ' ', pm.time_local)) AS timestamp,
            pm.latitude,
            pm.longitude,
            pm.sample_measurement AS pm25_level,
            t.sample_measurement AS temperature,
            w.sample_measurement AS wind_speed
        FROM
            `bigquery-public-data.epa_historical_air_quality.pm25_frm_hourly_summary` AS pm
        INNER JOIN
            `bigquery-public-data.epa_historical_air_quality.temperature_hourly_summary` AS t
            ON pm.state_code = t.state_code
            AND pm.county_code = t.county_code
            AND pm.site_num = t.site_num
            AND pm.date_local = t.date_local
            AND pm.time_local = t.time_local
        INNER JOIN
            `bigquery-public-data.epa_historical_air_quality.wind_hourly_summary` AS w
            ON pm.state_code = w.state_code
            AND pm.county_code = w.county_code
            AND pm.site_num = w.site_num
            AND pm.date_local = w.date_local
            AND pm.time_local = w.time_local
        WHERE
            pm.state_code = @state_code
            AND pm.county_code = @county_code
            AND pm.site_num = @site_num
    )
    SELECT * FROM raw_epa_data
    WHERE timestamp >= @buffer_time AND timestamp <= @end_time
    ORDER BY timestamp ASC;
    """
    
    query_job = client.query(query, job_config=job_config)
    epa_df = query_job.to_dataframe()
    
    if epa_df.empty:
        print("Error: No intersecting data found for PM2.5, Temperature, and Wind at this location/time.")
        return
        
    print(f"Retrieved {len(epa_df)} records. Processing time-series gaps...")
    
    # strict 1-hour intervals 
    # avg duplicate hours (multiple sensors/POCs), then resample and forward-fill gaps
    epa_df = epa_df.groupby('timestamp').mean().reset_index()
    epa_df = epa_df.set_index('timestamp').resample('h').ffill().reset_index()
    
    # 5. Extract cyclical features for the clean timeline
    DAYS_IN_YEAR = 365.2425
    HOURS_IN_DAY = 24.0
    day_of_year = epa_df['timestamp'].dt.dayofyear
    hour_of_day = epa_df['timestamp'].dt.hour
    
    epa_df['day_sin'] = np.sin(2 * np.pi * day_of_year / DAYS_IN_YEAR)
    epa_df['day_cos'] = np.cos(2 * np.pi * day_of_year / DAYS_IN_YEAR)
    epa_df['hour_sin'] = np.sin(2 * np.pi * hour_of_day / HOURS_IN_DAY)
    epa_df['hour_cos'] = np.cos(2 * np.pi * hour_of_day / HOURS_IN_DAY)
    
    lstm_features = ['pm25_level', 'latitude', 'longitude', 'temperature', 'wind_speed', 'day_sin', 'day_cos', 'hour_sin', 'hour_cos']
    
    scaled_features = scaler_x.transform(epa_df[lstm_features])
    
    sequence_length = 7
    
    current_sequence = scaled_features[0:sequence_length].copy()
    
    predictions_scaled = []
    y_true_scaled = []
    timestamps = []
    
    print(f"Executing recursive forecast for {len(scaled_features) - sequence_length} hours...")
    
    # 8. Loop through the remaining hours
    for i in range(sequence_length, len(scaled_features)):
        # Reshape current sequence for LSTM input (1 sample, 7 timesteps, 9 features)
        lstm_input = current_sequence.reshape(1, sequence_length, 9)
        
        # Predict the next hour's PM2.5 (outputs a scaled value)
        next_pm25_scaled = model.predict(lstm_input, verbose=0)[0][0]
        
        predictions_scaled.append(next_pm25_scaled)
        
        # Log the true target and timestamp for evaluation
        y_true_scaled.append(scaled_features[i][0])
        timestamps.append(epa_df['timestamp'].iloc[i])
        
        # TRUE weather/time features for hour 'i'
        # overwrite the PM2.5 feature (index 0) using SYNTHESIZED prediction.
        next_step_features = scaled_features[i].copy()
        next_step_features[0] = next_pm25_scaled 
        
        # Slide the window forward: drop the oldest row, append the synthesized row
        current_sequence = np.vstack([current_sequence[1:], next_step_features])
        
    y_true_scaled = np.array(y_true_scaled).reshape(-1, 1)
    predictions_scaled = np.array(predictions_scaled).reshape(-1, 1)
    
    # Inverse Transform back to real-world µg/m³
    y_pred_actual = scaler_y.inverse_transform(predictions_scaled).flatten()
    y_true_actual = scaler_y.inverse_transform(y_true_scaled).flatten()
    
    # plot
    mae = mean_absolute_error(y_true_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_true_actual, y_pred_actual))
    
    print(f"--- {days}-Day Recursive Window Evaluation ---")
    print(f"Location: State {state_code}, County {county_code}, Site {site_num}")
    print(f"Period: {start_time.date()} to {end_time.date()}")
    print(f"Window MAE:  {mae:.2f} µg/m³")
    print(f"Window RMSE: {rmse:.2f} µg/m³")
    
    plt.figure(figsize=(16, 6))
    plt.plot(timestamps, y_true_actual, label='Actual EPA PM2.5', color='blue', linewidth=2)
    plt.plot(timestamps, y_pred_actual, label='LSTM Recursive Forecast', color='red', linestyle='--', linewidth=2)
    
    plt.fill_between(timestamps, y_true_actual, y_pred_actual, color='gray', alpha=0.3, label='Deviation / Accumulated Error')
    
    plt.title(f'{days}-Day Recursive PM2.5 Forecast vs Real Data\nSite: {site_num} in County {county_code}', fontsize=14)
    plt.xlabel('Timestamp', fontsize=12)
    plt.ylabel('PM2.5 Concentration (µg/m³)', fontsize=12)
    plt.legend(loc='upper left', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
reliable_sites = find_reliable_sensor_sites(client, year='2023')
display(reliable_sites)

target_start = '2024-07-01 00:00:00' 

evaluate_epa_recursive_multi_day(
    client=client,
    model=lstm_model,
    scaler_x=scaler_x,       
    scaler_y=scaler_y,       
    state_code='19',         
    county_code='163',       
    site_num='0015',         
    start_date=target_start,
    days=10                  
)